# Workshop: real-gold 4D-STEM — browse, BF, DF, DPC, and 5 direct-ptycho kernels (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/bobleesj/a05a90185c6cddbb331342cae6d7e9c1/berk_workshop_v1.ipynb)

ONE notebook. Real gold from Hugging Face → load → browse → bright field → dark
field → DPC → all five direct-ptychography kernels (parallax, SSB, OBF, MF, ICOM)
→ side-by-side comparison.

Everything runs on torch on the Colab T4. Two installs only — `quantem.widget`
(TestPyPI prerelease) + `quantem` (`berk-workshop` branch on `bobleesj/quantem`).
No `quantem.live`.

**Total runtime on T4: ~3-4 min** (install dominates).

In [ ]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [ ]:
import quantem as em
import quantem.widget
import torch

# cuDNN grid_sample bug at these detector dims — disable for the DirectPtycho path.
torch.backends.cudnn.enabled = False

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__, "(cuDNN disabled)")
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")

In [ ]:
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.ascontiguousarray(np.load(os.path.join(asset, "data.npy")).astype(np.float32))
meta = json.load(open(os.path.join(asset, "meta.json")))

# Numpy-backed Dataset4dstem feeds both Show4DSTEM (via torch tensor) and
# DirectPtychography (which reads dataset.array internally).
dset = em.core.datastructures.Dataset4dstem.from_array(
    data, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset: shape {dset.shape}, dtype {dset.array.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad, CL {meta['camera_length_mm']} mm")

## Step 1 — Browse the 4D-STEM dataset interactively

Drag the scan cursor; CBED updates live. This is real-time per-scan-position BF/DF.

In [ ]:
# Build a torch-backed view for Show4DSTEM so cursor drag is GPU-fast.
dset_torch = em.core.datastructures.Dataset4dstem.from_tensor(
    torch.from_numpy(data).to("cuda" if torch.cuda.is_available() else "cpu"),
    sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
quantem.widget.Show4DSTEM(dset_torch)

## Step 2 — Bright field, dark field, DPC (inline torch on GPU)

Hardcode aperture at the detector center. One torch reduction per panel.

In [ ]:
data_f = torch.from_numpy(data).to("cuda" if torch.cuda.is_available() else "cpu")

# Detector grid + hardcoded aperture center (geometric).
H, W = data_f.shape[-2:]
cy, cx = H / 2, W / 2
row = torch.arange(H, device=data_f.device, dtype=torch.float32)[:, None]
col = torch.arange(W, device=data_f.device, dtype=torch.float32)[None, :]
rr, cc = torch.meshgrid(row.squeeze(), col.squeeze(), indexing="ij")
r_from_center = ((rr - cy) ** 2 + (cc - cx) ** 2).sqrt()

# BF + DF
BF_RADIUS_PX = 6.0
bf_mask = (r_from_center <= BF_RADIUS_PX).float()
df_mask = 1.0 - bf_mask
bf = (data_f * bf_mask).sum(dim=(-2, -1)).cpu().numpy()
df = (data_f * df_mask).sum(dim=(-2, -1)).cpu().numpy()

# CoM / DPC — per-scan-position centroid (qx, qy) → row, col deflection + magnitude
qx = row.expand(H, W)
qy = col.expand(H, W)
total_per_dp = data_f.sum(dim=(-2, -1))
com_row = (data_f * qx).sum(dim=(-2, -1)) / total_per_dp
com_col = (data_f * qy).sum(dim=(-2, -1)) / total_per_dp
com_row -= com_row.mean()
com_col -= com_col.mean()
com_mag = (com_row ** 2 + com_col ** 2).sqrt()

print(f"BF range  [{bf.min():.1f}, {bf.max():.1f}]")
print(f"DF range  [{df.min():.1f}, {df.max():.1f}]")
print(f"|CoM| max  {com_mag.max().item():.4f} px")

### BF + DF side by side

In [ ]:
quantem.widget.Show2D(
    [bf, df],
    labels=["Bright field", "Dark field"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
)

### DPC — CoM row + CoM col + |CoM|

In [ ]:
quantem.widget.Show2D(
    [com_row.cpu().numpy(), com_col.cpu().numpy(), com_mag.cpu().numpy()],
    labels=["CoM row (qx)", "CoM col (qy)", "|CoM| total"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="RdBu_r",
)

## Step 3 — DirectPtychography — five single-shot kernels

`DirectPtychography` runs CoM + origin fit + auto-rotation, then a single forward
pass to recover phase via one of five deconvolution kernels:

- **`parallax`** — parallax / tilt approximation
- **`ssb`** — single-sideband (a.k.a. aberration-corrected bright field)
- **`obf`** — optimum bright field
- **`mf`** — matched filter
- **`icom`** — integrated CoM

Build once, sweep all five kernels.

In [ ]:
from quantem.diffractive_imaging import DirectPtychography

direct = DirectPtychography.from_dataset4d(
    dset,
    energy=meta["voltage_kV"] * 1e3,                      # 300 kV -> 300000 eV
    semiangle_cutoff=meta["probe_semiangle_mrad"] * 1e-3, # 30 mrad -> 0.030 rad
    rotation_angle=None,                                   # auto-estimate
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)
print(f"DirectPtychography built on {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
import time
KERNELS = ["parallax", "ssb", "obf", "mf", "icom"]
phases = {}
for k in KERNELS:
    t0 = time.time()
    direct.reconstruct(deconvolution_kernel=k, verbose=False)
    phases[k] = direct.corrected_bf.detach().cpu().numpy()
    print(f"  {k:>10}: {time.time()-t0:.2f}s, range [{phases[k].min():.2f}, {phases[k].max():.2f}]")

## Step 4 — Compare all five kernels side by side

Same data, same forward pass, different deconvolution kernels — different
contrast / resolution tradeoffs.

In [ ]:
quantem.widget.Show2D(
    [phases[k] for k in KERNELS],
    labels=[k for k in KERNELS],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
)

## Step 5 — Why does this matter? Phase retrieval vs classic imaging

Look at the panels below side by side, in this order: **BF · DF · |CoM| · parallax · SSB**.

- **BF / DF** = intensity contrast. Gold lattice fringes mostly washed out — the contrast is whatever survives integration over the BF disk (or its complement). Dose-efficient but resolution-limited by the disk size.
- **|CoM|** = first-moment information per scan position. Sees first-order electric-field deflection; better than BF/DF but still a single scalar per position.
- **`parallax`, `SSB`** = phase retrieval. Each scan position contributes its full diffraction pattern; the kernel deconvolves the probe-transfer function to recover the **complex object phase**. Result: atomic-lattice fringes, sharper edges, less dose for equivalent SNR.

This is the workshop's scientific punchline: phase retrieval recovers contrast + resolution that BF/DF integration physically can't access.

In [ ]:
quantem.widget.Show2D(
    [bf, df, com_mag.cpu().numpy(), phases["parallax"], phases["ssb"]],
    labels=["BF", "DF", "|CoM|", "parallax", "SSB"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
)

## What you just did

1. Loaded real 4D-STEM gold from Hugging Face → torch GPU + numpy `Dataset4dstem`.
2. Browsed it with `Show4DSTEM`.
3. Computed BF / DF / DPC (CoM_row, CoM_col, |CoM|) inline in torch.
4. Built `DirectPtychography` and swept five deconvolution kernels (parallax, SSB, OBF, MF, ICOM) in seconds each.
5. Compared all the modalities side by side: classic imaging (BF/DF) vs first-moment (|CoM|) vs phase retrieval (parallax, SSB).

## Why phase retrieval beats BF/DF — the workshop takeaway

| Method | Information used | Contrast mechanism | Resolution ceiling |
|---|---|---|---|
| BF, DF | total counts inside / outside the BF disk | intensity (atomic Z, thickness) | ~probe size; integration smears |
| DPC (|CoM|) | first moment of each CBED | first-order field deflection | better than BF, still scalar/pixel |
| Phase retrieval (parallax, SSB, OBF, MF, ICOM) | the **full** CBED at every scan position | recovers the complex object phase | sub-Ångström possible |

Single forward pass on T4 — every kernel finishes in seconds. The phase image
shows atomic-lattice fringes that BF/DF can't physically resolve at the same dose.

## Try next

- Swap to `gold_512_npy_bin4` for a 4× finer detector — sharpest phase images.
- Pass explicit `rotation_angle=` to `DirectPtychography.from_dataset4d` if the auto-estimate is off.
- v2 will add iterative ptychography (`PtychoLite`) — even higher-resolution phase, multi-slice support, refinement on top of these single-shot kernels.